# From Scratch Object Detection Challenge - FCOS v4

This is a time-safe stronger Kaggle notebook than the CenterNet baseline. It implements
an FCOS/RetinaNet-style detector from scratch:

- ImageNet-pretrained ResNet50 backbone.
- Custom FPN neck.
- Dense classification, box-distance, and centerness heads.
- FCOS-style point assignment, focal loss, GIoU loss, centerness loss.
- Class-wise NMS, validation mAP@0.5 sweep, and horizontal-flip TTA for test.
- Shorter training with early stop, so Kaggle has time to write the submission.

The detector head, assignment, losses, decoding, NMS, threshold sweep, and
submission writer are all implemented in this notebook.

In [1]:
from __future__ import annotations

import csv
import json
import math
import random
import time
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageEnhance
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu count", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"gpu {i}", torch.cuda.get_device_name(i))

torch 2.10.0+cu128
cuda available True
gpu count 2
gpu 0 Tesla T4
gpu 1 Tesla T4


## Config

In [2]:
SEED = 42
IMAGE_SIZE = 640
EPOCHS = 16
MIN_EPOCHS = 8
EARLY_STOP_PATIENCE = 6
BATCH_SIZE = 16
NUM_WORKERS = 0
LR = 1.5e-4
WEIGHT_DECAY = 1e-4
PRETRAINED_BACKBONE = True

VAL_CONF_THRESH = 0.03
PRED_CONF_THRESH = 0.05
NMS_THRESH = 0.55
TOPK_CANDIDATES = 1500
MAX_DET = 300

RUN_DIR = Path("/kaggle/working/fcos_v4_resnet50")
RUN_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count() if DEVICE.type == "cuda" else 0


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/from-scratch-object-detection-challenge/datasets"),
        Path("/kaggle/input/from-scratch-object-detection-challenge/datasets"),
        Path("/kaggle/input/from-scratch-object-detection-challenge"),
        Path("/kaggle/working/datasets"),
        Path("./datasets"),
    ]
    for candidate in candidates:
        if (candidate / "classes.json").exists():
            return candidate
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.rglob("classes.json"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find classes.json. Add the competition dataset to notebook inputs.")


def make_grad_scaler():
    if DEVICE.type != "cuda":
        return torch.cuda.amp.GradScaler(enabled=False)
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)


def autocast_context():
    if DEVICE.type != "cuda":
        return nullcontext()
    try:
        return torch.amp.autocast("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=True)


DATA_ROOT = find_data_root()
print("DATA_ROOT =", DATA_ROOT)
print("RUN_DIR =", RUN_DIR)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_ROOT = /kaggle/input/competitions/from-scratch-object-detection-challenge/datasets
RUN_DIR = /kaggle/working/fcos_v4_resnet50


## Dataset Parser and Augmentations

In [3]:
DEFAULT_CLASSES = ["person", "car", "dog", "cat", "chair"]


@dataclass
class Sample:
    image_id: str
    boxes: np.ndarray
    labels: np.ndarray


def normalize_classes(raw):
    if isinstance(raw, dict):
        if "root" in raw and isinstance(raw["root"], list):
            return [str(item) for item in raw["root"]]
        if all(isinstance(v, int) for v in raw.values()):
            return [name for name, _ in sorted(raw.items(), key=lambda item: item[1])]
        if all(isinstance(k, str) and k.isdigit() for k in raw):
            return [raw[str(i)] for i in range(len(raw))]
    if isinstance(raw, list):
        return [str(item) for item in raw]
    return list(DEFAULT_CLASSES)


def load_classes(data_root: str | Path) -> list[str]:
    path = Path(data_root) / "classes.json"
    if not path.exists():
        return list(DEFAULT_CLASSES)
    with path.open("r", encoding="utf-8") as f:
        return normalize_classes(json.load(f))


def _parse_box(raw: Any) -> list[float] | None:
    if isinstance(raw, dict):
        keys = raw.keys()
        if {"x_min", "y_min", "x_max", "y_max"}.issubset(keys):
            return [raw["x_min"], raw["y_min"], raw["x_max"], raw["y_max"]]
        if {"xmin", "ymin", "xmax", "ymax"}.issubset(keys):
            return [raw["xmin"], raw["ymin"], raw["xmax"], raw["ymax"]]
        if {"x", "y", "width", "height"}.issubset(keys):
            x, y, w, h = raw["x"], raw["y"], raw["width"], raw["height"]
            return [x, y, x + w, y + h]
    if isinstance(raw, (list, tuple)) and len(raw) == 4:
        return [float(v) for v in raw]
    return None


def _iter_annotation_objects(data: Any):
    if isinstance(data, dict) and "annotations" in data:
        yield from _iter_annotation_objects(data["annotations"])
    elif isinstance(data, dict):
        for value in data.values():
            yield from _iter_annotation_objects(value)
    elif isinstance(data, list):
        for item in data:
            if isinstance(item, dict):
                if _parse_box(item.get("bbox", item.get("box", item))) is not None:
                    yield item
                else:
                    for key in ("objects", "annotations", "boxes", "bboxes", "bounding_boxes"):
                        if isinstance(item.get(key), list):
                            yield from _iter_annotation_objects(item[key])


def _infer_numeric_label_offset(data: Any, num_classes: int) -> int:
    values = []
    for obj in _iter_annotation_objects(data):
        for key in ("class", "label", "category", "category_name", "class_name", "category_id"):
            if key in obj and isinstance(obj[key], (int, np.integer)):
                values.append(int(obj[key]))
                break
    if values and min(values) >= 1 and max(values) <= num_classes:
        return 1
    return 0


def _label_to_index(value: Any, class_to_idx: dict[str, int], label_offset: int = 0) -> int | None:
    if isinstance(value, str):
        return class_to_idx.get(value.lower())
    if isinstance(value, (int, np.integer)):
        idx = int(value) - label_offset
        if 0 <= idx < len(class_to_idx):
            return idx
    return None


def _parse_object(obj: dict[str, Any], class_to_idx: dict[str, int], label_offset: int = 0):
    box = None
    for key in ("bbox", "box", "bounding_box"):
        if key in obj:
            box = _parse_box(obj[key])
            break
    if box is None:
        box = _parse_box(obj)
    if box is None:
        return None

    label_value = None
    for key in ("class", "label", "category", "category_name", "class_name", "category_id"):
        if key in obj:
            label_value = obj[key]
            break
    label = _label_to_index(label_value, class_to_idx, label_offset)
    if label is None:
        return None

    x1, y1, x2, y2 = [float(v) for v in box]
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2, y2], label


def _objects_from_sample(item: dict[str, Any]):
    if isinstance(item.get("boxes"), list) and isinstance(item.get("labels"), list):
        return [{"bbox": box, "class": label} for box, label in zip(item["boxes"], item["labels"])]
    for key in ("objects", "annotations", "boxes", "bboxes", "bounding_boxes"):
        value = item.get(key)
        if isinstance(value, list):
            return value
    return []


def _image_id_from_item(item: dict[str, Any]) -> str | None:
    for key in ("image_id", "file_name", "filename", "image", "path"):
        if item.get(key) is not None:
            return str(item[key])
    return None


def _sample_from_objects(image_id: str, objects, class_to_idx: dict[str, int], label_offset: int) -> Sample:
    boxes, labels = [], []
    for obj in objects:
        if not isinstance(obj, dict):
            continue
        parsed = _parse_object(obj, class_to_idx, label_offset)
        if parsed is None:
            continue
        box, label = parsed
        boxes.append(box)
        labels.append(label)
    return Sample(
        image_id=str(image_id),
        boxes=np.asarray(boxes, dtype=np.float32).reshape(-1, 4),
        labels=np.asarray(labels, dtype=np.int64),
    )


def load_annotations(path: str | Path, classes: list[str]) -> list[Sample]:
    path = Path(path)
    class_to_idx = {name.lower(): i for i, name in enumerate(classes)}
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    label_offset = _infer_numeric_label_offset(data, len(classes))

    if isinstance(data, dict) and "images" in data and "annotations" in data:
        if isinstance(data.get("categories"), list):
            category_to_name = {
                item.get("id"): item.get("name")
                for item in data["categories"]
                if isinstance(item, dict) and "id" in item and "name" in item
            }
            for ann in data["annotations"]:
                if "category_id" in ann and ann["category_id"] in category_to_name:
                    ann.setdefault("class", category_to_name[ann["category_id"]])
        id_to_name = {
            item.get("id", item.get("image_id")): item.get("file_name", item.get("filename"))
            for item in data["images"]
        }
        grouped = {str(name): [] for name in id_to_name.values() if name}
        for ann in data["annotations"]:
            image_key = ann.get("image_id")
            image_id = id_to_name.get(image_key, image_key)
            if image_id is not None:
                grouped.setdefault(str(image_id), []).append(ann)
        return [_sample_from_objects(image_id, objects, class_to_idx, label_offset) for image_id, objects in grouped.items()]

    if isinstance(data, dict) and "annotations" in data:
        data = data["annotations"]

    if isinstance(data, list):
        flat_rows = [item for item in data if isinstance(item, dict) and _parse_box(item.get("bbox", item.get("box", item))) is not None]
        if flat_rows:
            grouped = {}
            for item in flat_rows:
                image_id = _image_id_from_item(item)
                if image_id:
                    grouped.setdefault(image_id, []).append(item)
            return [_sample_from_objects(image_id, objects, class_to_idx, label_offset) for image_id, objects in grouped.items()]
        samples = []
        for item in data:
            if not isinstance(item, dict):
                continue
            image_id = _image_id_from_item(item)
            if image_id:
                samples.append(_sample_from_objects(image_id, _objects_from_sample(item), class_to_idx, label_offset))
        return samples

    if isinstance(data, dict):
        samples = []
        for image_id, value in data.items():
            if isinstance(value, dict):
                objects = _objects_from_sample(value)
            elif isinstance(value, list):
                objects = value
            else:
                objects = []
            samples.append(_sample_from_objects(str(image_id), objects, class_to_idx, label_offset))
        return samples

    raise ValueError(f"Unsupported annotation format in {path}")


def _find_image(image_root: Path, image_id: str) -> Path:
    direct = image_root / str(image_id)
    if direct.exists():
        return direct
    basename = Path(str(image_id)).name
    direct = image_root / basename
    if direct.exists():
        return direct
    stem = Path(basename).stem
    for suffix in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
        candidate = image_root / f"{stem}{suffix}"
        if candidate.exists():
            return candidate
    return image_root / basename


def _image_to_tensor(image: Image.Image) -> torch.Tensor:
    arr = np.asarray(image, dtype=np.float32) / 255.0
    arr = (arr - np.asarray([0.485, 0.456, 0.406], dtype=np.float32)) / np.asarray(
        [0.229, 0.224, 0.225], dtype=np.float32
    )
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()


def _letterbox(image: Image.Image, boxes: np.ndarray, image_size: int):
    orig_w, orig_h = image.size
    scale = image_size / max(orig_w, orig_h)
    new_w = int(round(orig_w * scale))
    new_h = int(round(orig_h * scale))
    image = image.resize((new_w, new_h), Image.BILINEAR)
    canvas = Image.new("RGB", (image_size, image_size), (114, 114, 114))
    pad_x = (image_size - new_w) // 2
    pad_y = (image_size - new_h) // 2
    canvas.paste(image, (pad_x, pad_y))
    boxes = boxes.copy()
    if len(boxes):
        boxes[:, [0, 2]] = boxes[:, [0, 2]] * scale + pad_x
        boxes[:, [1, 3]] = boxes[:, [1, 3]] * scale + pad_y
        boxes[:, 0::2] = boxes[:, 0::2].clip(0, image_size - 1)
        boxes[:, 1::2] = boxes[:, 1::2].clip(0, image_size - 1)
    meta = {"orig_h": orig_h, "orig_w": orig_w, "scale": scale, "pad_x": pad_x, "pad_y": pad_y}
    return canvas, boxes, meta


class DetectionDataset(Dataset):
    def __init__(self, data_root: str | Path, split: str, image_size: int = 640, augment: bool = False, classes=None):
        self.data_root = Path(data_root)
        self.split = split
        self.image_size = image_size
        self.augment = augment
        self.classes = classes or load_classes(data_root)
        self.samples = load_annotations(self.data_root / "annotations" / f"{split}.json", self.classes)
        self.image_root = self.data_root / split / "images"

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        path = _find_image(self.image_root, sample.image_id)
        image = Image.open(path).convert("RGB")
        boxes = sample.boxes.copy()
        labels = sample.labels.copy()
        image, boxes, meta = _letterbox(image, boxes, self.image_size)

        if self.augment:
            if len(boxes) and random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                old_x1 = boxes[:, 0].copy()
                old_x2 = boxes[:, 2].copy()
                boxes[:, 0] = self.image_size - old_x2
                boxes[:, 2] = self.image_size - old_x1
            if random.random() < 0.7:
                image = ImageEnhance.Color(image).enhance(random.uniform(0.75, 1.30))
                image = ImageEnhance.Contrast(image).enhance(random.uniform(0.75, 1.30))
                image = ImageEnhance.Brightness(image).enhance(random.uniform(0.80, 1.25))

        return _image_to_tensor(image), {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.long),
            "image_id": Path(sample.image_id).name,
            "orig_size": torch.tensor([meta["orig_h"], meta["orig_w"]], dtype=torch.long),
            "scale_pad": torch.tensor([meta["scale"], meta["pad_x"], meta["pad_y"]], dtype=torch.float32),
        }


class InferenceImageDataset(Dataset):
    def __init__(self, data_root: str | Path, split: str = "test", image_size: int = 640):
        self.data_root = Path(data_root)
        self.split = split
        self.image_size = image_size
        self.image_root = self.data_root / split / "images"
        self.paths = sorted([
            path
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
            for path in self.image_root.glob(ext)
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]
        image = Image.open(path).convert("RGB")
        empty_boxes = np.zeros((0, 4), dtype=np.float32)
        image, _boxes, meta = _letterbox(image, empty_boxes, self.image_size)
        return _image_to_tensor(image), {
            "image_id": path.name,
            "orig_size": torch.tensor([meta["orig_h"], meta["orig_w"]], dtype=torch.long),
            "scale_pad": torch.tensor([meta["scale"], meta["pad_x"], meta["pad_y"]], dtype=torch.float32),
        }


def collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images, dim=0), list(targets)

In [4]:
CLASSES = load_classes(DATA_ROOT)
print("classes:", CLASSES)
train_samples = load_annotations(DATA_ROOT / "annotations" / "train.json", CLASSES)
val_samples = load_annotations(DATA_ROOT / "annotations" / "val.json", CLASSES)
print("train samples:", len(train_samples), "val samples:", len(val_samples))
non_empty = next((s for s in train_samples if len(s.boxes)), None)
if non_empty:
    print("example:", non_empty.image_id, non_empty.boxes[:2].tolist(), non_empty.labels[:2].tolist())

classes: ['person', 'car', 'dog', 'cat', 'chair']
train samples: 7500 val samples: 1500
example: train/images/img_000c52c6d12f.jpg [[1.0, 1.0, 500.0, 375.0]] [2]


## Model: ResNet50 FPN + FCOS Heads

In [5]:
def conv_gn_relu(in_ch: int, out_ch: int, k: int = 3):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=k // 2, bias=False),
        nn.GroupNorm(16, out_ch),
        nn.ReLU(inplace=True),
    )


class Scale(nn.Module):
    def __init__(self, init_value: float = 1.0):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(float(init_value)))

    def forward(self, x):
        return x * self.scale


class FCOSResNet50FPN(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = True, channels: int = 128):
        super().__init__()
        from torchvision.models import resnet50

        try:
            from torchvision.models import ResNet50_Weights
            weights = ResNet50_Weights.DEFAULT if pretrained else None
            base = resnet50(weights=weights)
        except Exception as exc:
            if pretrained:
                print("pretrained backbone unavailable, using random init:", repr(exc))
            try:
                base = resnet50(weights=None)
            except TypeError:
                base = resnet50(pretrained=False)

        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.lat3 = nn.Conv2d(512, channels, kernel_size=1)
        self.lat4 = nn.Conv2d(1024, channels, kernel_size=1)
        self.lat5 = nn.Conv2d(2048, channels, kernel_size=1)
        self.p3 = conv_gn_relu(channels, channels)
        self.p4 = conv_gn_relu(channels, channels)
        self.p5 = conv_gn_relu(channels, channels)
        self.p6 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1, bias=False),
            nn.GroupNorm(16, channels),
            nn.ReLU(inplace=True),
        )

        tower = []
        for _ in range(4):
            tower.append(conv_gn_relu(channels, channels))
        self.cls_tower = nn.Sequential(*tower)
        tower = []
        for _ in range(4):
            tower.append(conv_gn_relu(channels, channels))
        self.box_tower = nn.Sequential(*tower)

        self.cls_logits = nn.Conv2d(channels, num_classes, kernel_size=3, padding=1)
        self.bbox_pred = nn.Conv2d(channels, 4, kernel_size=3, padding=1)
        self.centerness = nn.Conv2d(channels, 1, kernel_size=3, padding=1)
        self.scales = nn.ModuleList([Scale(1.0) for _ in range(4)])

        prior_prob = 0.01
        nn.init.constant_(self.cls_logits.bias, -math.log((1 - prior_prob) / prior_prob))

    def forward(self, x):
        x = self.stem(x)
        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)

        p5 = self.lat5(c5)
        p4 = self.lat4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="nearest")
        p3 = self.lat3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="nearest")
        features = [self.p3(p3), self.p4(p4), self.p5(p5)]
        features.append(self.p6(features[-1]))

        cls_outputs, box_outputs, ctr_outputs = [], [], []
        for level, feat in enumerate(features):
            cls_feat = self.cls_tower(feat)
            box_feat = self.box_tower(feat)
            cls_outputs.append(self.cls_logits(cls_feat))
            box_outputs.append(F.relu(self.scales[level](self.bbox_pred(box_feat))))
            ctr_outputs.append(self.centerness(box_feat))
        return {"cls": cls_outputs, "box": box_outputs, "ctr": ctr_outputs}

## FCOS Assignment, Loss, Decode, NMS

In [6]:
STRIDES = [8, 16, 32, 64]
REG_RANGES = [(0, 96), (64, 192), (128, 384), (256, 1e8)]


def make_points(outputs, image_size: int, device: torch.device):
    all_points, all_strides, all_ranges, shapes = [], [], [], []
    for out, stride, reg_range in zip(outputs["cls"], STRIDES, REG_RANGES):
        h, w = out.shape[-2:]
        ys = (torch.arange(h, device=device, dtype=torch.float32) + 0.5) * stride
        xs = (torch.arange(w, device=device, dtype=torch.float32) + 0.5) * stride
        yy, xx = torch.meshgrid(ys, xs, indexing="ij")
        points = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
        all_points.append(points)
        all_strides.append(torch.full((points.shape[0],), stride, device=device))
        all_ranges.append(torch.tensor(reg_range, device=device, dtype=torch.float32).view(1, 2).expand(points.shape[0], 2))
        shapes.append((h, w))
    return torch.cat(all_points), torch.cat(all_strides), torch.cat(all_ranges), shapes


def flatten_outputs(outputs):
    cls = torch.cat([x.permute(0, 2, 3, 1).reshape(x.shape[0], -1, x.shape[1]) for x in outputs["cls"]], dim=1)
    box = torch.cat([x.permute(0, 2, 3, 1).reshape(x.shape[0], -1, 4) for x in outputs["box"]], dim=1)
    ctr = torch.cat([x.permute(0, 2, 3, 1).reshape(x.shape[0], -1) for x in outputs["ctr"]], dim=1)
    return cls, box, ctr


def encode_targets(targets, points, strides, reg_ranges, num_classes: int):
    batch = len(targets)
    n = points.shape[0]
    labels = torch.full((batch, n), -1, dtype=torch.long, device=points.device)
    reg_targets = torch.zeros((batch, n, 4), dtype=torch.float32, device=points.device)
    ctr_targets = torch.zeros((batch, n), dtype=torch.float32, device=points.device)

    xs, ys = points[:, 0], points[:, 1]
    for b, target in enumerate(targets):
        boxes = target["boxes"].to(points.device)
        gt_labels = target["labels"].to(points.device)
        if len(boxes) == 0:
            continue
        l = xs[:, None] - boxes[None, :, 0]
        t = ys[:, None] - boxes[None, :, 1]
        r = boxes[None, :, 2] - xs[:, None]
        btm = boxes[None, :, 3] - ys[:, None]
        reg = torch.stack([l, t, r, btm], dim=2)
        inside_box = reg.min(dim=2).values > 0

        centers = (boxes[:, :2] + boxes[:, 2:]) * 0.5
        radius = strides[:, None] * 1.5
        center_x1 = torch.maximum(boxes[None, :, 0], centers[None, :, 0] - radius)
        center_y1 = torch.maximum(boxes[None, :, 1], centers[None, :, 1] - radius)
        center_x2 = torch.minimum(boxes[None, :, 2], centers[None, :, 0] + radius)
        center_y2 = torch.minimum(boxes[None, :, 3], centers[None, :, 1] + radius)
        inside_center = (xs[:, None] >= center_x1) & (xs[:, None] <= center_x2) & (ys[:, None] >= center_y1) & (ys[:, None] <= center_y2)

        max_reg = reg.max(dim=2).values
        in_range = (max_reg >= reg_ranges[:, None, 0]) & (max_reg <= reg_ranges[:, None, 1])
        areas = ((boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]))[None, :].expand(n, len(boxes)).clone()
        areas[~(inside_box & inside_center & in_range)] = 1e8
        min_area, min_inds = areas.min(dim=1)
        pos = min_area < 1e8
        matched = min_inds[pos]
        labels[b, pos] = gt_labels[matched]
        reg_targets[b, pos] = reg[pos, matched] / strides[pos, None]
        left_right = reg_targets[b, pos][:, [0, 2]]
        top_bottom = reg_targets[b, pos][:, [1, 3]]
        ctr_targets[b, pos] = torch.sqrt(
            (left_right.min(dim=1).values / left_right.max(dim=1).values.clamp(min=1e-6))
            * (top_bottom.min(dim=1).values / top_bottom.max(dim=1).values.clamp(min=1e-6))
        ).clamp(0, 1)
    return labels, reg_targets, ctr_targets


def sigmoid_focal_loss(logits, targets, alpha: float = 0.25, gamma: float = 2.0, reduction: str = "sum"):
    prob = logits.sigmoid()
    ce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = prob * targets + (1 - prob) * (1 - targets)
    loss = ce_loss * ((1 - p_t) ** gamma)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * loss
    return loss.sum() if reduction == "sum" else loss.mean()


def distances_to_boxes(points, distances, strides=None):
    if strides is not None:
        distances = distances * strides[:, None]
    x1 = points[:, 0] - distances[:, 0]
    y1 = points[:, 1] - distances[:, 1]
    x2 = points[:, 0] + distances[:, 2]
    y2 = points[:, 1] + distances[:, 3]
    return torch.stack([x1, y1, x2, y2], dim=1)


def box_iou_matrix(boxes1, boxes2):
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), device=boxes1.device)
    lt = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
    union = area1[:, None] + area2[None, :] - inter
    return inter / union.clamp(min=1e-6)


def giou_loss(pred_boxes, target_boxes):
    lt = torch.maximum(pred_boxes[:, :2], target_boxes[:, :2])
    rb = torch.minimum(pred_boxes[:, 2:], target_boxes[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, 0] * wh[:, 1]
    area_p = (pred_boxes[:, 2] - pred_boxes[:, 0]).clamp(min=0) * (pred_boxes[:, 3] - pred_boxes[:, 1]).clamp(min=0)
    area_t = (target_boxes[:, 2] - target_boxes[:, 0]).clamp(min=0) * (target_boxes[:, 3] - target_boxes[:, 1]).clamp(min=0)
    union = area_p + area_t - inter
    iou = inter / union.clamp(min=1e-6)
    enclose_lt = torch.minimum(pred_boxes[:, :2], target_boxes[:, :2])
    enclose_rb = torch.maximum(pred_boxes[:, 2:], target_boxes[:, 2:])
    enclose_wh = (enclose_rb - enclose_lt).clamp(min=0)
    area_c = enclose_wh[:, 0] * enclose_wh[:, 1]
    giou = iou - (area_c - union) / area_c.clamp(min=1e-6)
    return 1 - giou


def detection_loss(outputs, targets, image_size: int):
    cls_logits, box_pred, ctr_logits = flatten_outputs(outputs)
    points, strides, reg_ranges, _ = make_points(outputs, image_size, cls_logits.device)
    labels, reg_targets, ctr_targets = encode_targets(targets, points, strides, reg_ranges, cls_logits.shape[-1])
    pos = labels >= 0
    num_pos = pos.sum().clamp(min=1).float()

    cls_targets = torch.zeros_like(cls_logits)
    pos_b, pos_i = torch.where(pos)
    cls_targets[pos_b, pos_i, labels[pos]] = 1.0
    cls_loss = sigmoid_focal_loss(cls_logits, cls_targets, reduction="sum") / num_pos

    if pos.sum() == 0:
        box_loss = box_pred.sum() * 0
        ctr_loss = ctr_logits.sum() * 0
    else:
        points_pos = points[pos_i]
        strides_pos = strides[pos_i]
        pred_boxes = distances_to_boxes(points_pos, box_pred[pos_b, pos_i], strides_pos)
        target_boxes = distances_to_boxes(points_pos, reg_targets[pos_b, pos_i], strides_pos)
        weights = ctr_targets[pos].detach()
        box_loss = (giou_loss(pred_boxes, target_boxes) * weights).sum() / weights.sum().clamp(min=1e-6)
        ctr_loss = F.binary_cross_entropy_with_logits(ctr_logits[pos], ctr_targets[pos], reduction="sum") / num_pos

    total = cls_loss + 2.0 * box_loss + ctr_loss
    return {"loss": total, "cls_loss": cls_loss.detach(), "box_loss": box_loss.detach(), "ctr_loss": ctr_loss.detach(), "num_pos": num_pos.detach()}


def nms(boxes, scores, iou_threshold: float):
    if boxes.numel() == 0:
        return torch.empty((0,), dtype=torch.long, device=boxes.device)
    order = scores.argsort(descending=True)
    keep = []
    while order.numel() > 0:
        current = order[0]
        keep.append(current)
        if order.numel() == 1:
            break
        ious = box_iou_matrix(boxes[current].unsqueeze(0), boxes[order[1:]]).squeeze(0)
        order = order[1:][ious <= iou_threshold]
    return torch.stack(keep)


def classwise_nms(boxes, scores, labels, iou_threshold: float):
    keep = []
    for label in labels.unique():
        inds = torch.where(labels == label)[0]
        keep.append(inds[nms(boxes[inds], scores[inds], iou_threshold)])
    if not keep:
        return torch.empty((0,), dtype=torch.long, device=boxes.device)
    keep = torch.cat(keep)
    return keep[scores[keep].argsort(descending=True)]


@torch.no_grad()
def decode_detections(outputs, image_size: int, conf_thresh: float, nms_thresh: float, topk: int = 1500, max_det: int = 300):
    cls_logits, box_pred, ctr_logits = flatten_outputs(outputs)
    points, strides, _ranges, _shapes = make_points(outputs, image_size, cls_logits.device)
    probs = cls_logits.sigmoid() * ctr_logits.sigmoid().unsqueeze(-1).sqrt()
    results = []
    for b in range(cls_logits.shape[0]):
        scores, flat_inds = torch.topk(probs[b].reshape(-1), k=min(topk, probs.shape[1] * probs.shape[2]))
        keep = scores >= conf_thresh
        if keep.sum() == 0:
            results.append({"boxes": torch.empty((0, 4), device=cls_logits.device), "scores": torch.empty((0,), device=cls_logits.device), "labels": torch.empty((0,), dtype=torch.long, device=cls_logits.device)})
            continue
        scores = scores[keep]
        flat_inds = flat_inds[keep]
        point_inds = flat_inds // probs.shape[2]
        labels = flat_inds % probs.shape[2]
        boxes = distances_to_boxes(points[point_inds], box_pred[b, point_inds], strides[point_inds])
        boxes[:, 0::2] = boxes[:, 0::2].clamp(0, image_size - 1)
        boxes[:, 1::2] = boxes[:, 1::2].clamp(0, image_size - 1)
        valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
        boxes, scores, labels = boxes[valid], scores[valid], labels[valid]
        chosen = classwise_nms(boxes, scores, labels, nms_thresh)[:max_det]
        results.append({"boxes": boxes[chosen], "scores": scores[chosen], "labels": labels[chosen]})
    return results

## mAP@0.5 Validation

In [7]:
def average_precision(recalls: np.ndarray, precisions: np.ndarray) -> float:
    recalls = np.concatenate([[0.0], recalls, [1.0]])
    precisions = np.concatenate([[0.0], precisions, [0.0]])
    for i in range(len(precisions) - 1, 0, -1):
        precisions[i - 1] = max(precisions[i - 1], precisions[i])
    change = np.where(recalls[1:] != recalls[:-1])[0]
    return float(np.sum((recalls[change + 1] - recalls[change]) * precisions[change + 1]))


def map_at_iou(predictions: list[dict], targets: list[dict], num_classes: int, iou_threshold: float = 0.5) -> float:
    aps = []
    for cls in range(num_classes):
        pred_rows = []
        gt_by_image, matched = {}, {}
        for target in targets:
            image_id = str(target["image_id"])
            cls_boxes = target["boxes"][target["labels"] == cls]
            gt_by_image[image_id] = cls_boxes
            matched[image_id] = np.zeros((len(cls_boxes),), dtype=bool)
        for pred in predictions:
            image_id = str(pred["image_id"])
            for box, score in zip(pred["boxes"][pred["labels"] == cls], pred["scores"][pred["labels"] == cls]):
                pred_rows.append((image_id, float(score), box))
        total_gt = sum(len(v) for v in gt_by_image.values())
        if total_gt == 0:
            continue
        if not pred_rows:
            aps.append(0.0)
            continue
        pred_rows.sort(key=lambda row: row[1], reverse=True)
        tp = np.zeros((len(pred_rows),), dtype=np.float32)
        fp = np.zeros((len(pred_rows),), dtype=np.float32)
        for i, (image_id, _score, box) in enumerate(pred_rows):
            gt_boxes = gt_by_image.get(image_id)
            if gt_boxes is None or len(gt_boxes) == 0:
                fp[i] = 1
                continue
            ious = box_iou_matrix(box.unsqueeze(0), gt_boxes.to(box.device)).squeeze(0).cpu().numpy()
            best = int(np.argmax(ious))
            if ious[best] >= iou_threshold and not matched[image_id][best]:
                tp[i] = 1
                matched[image_id][best] = True
            else:
                fp[i] = 1
        tp_cum = np.cumsum(tp)
        fp_cum = np.cumsum(fp)
        recalls = tp_cum / max(float(total_gt), 1e-6)
        precisions = tp_cum / np.maximum(tp_cum + fp_cum, 1e-6)
        aps.append(average_precision(recalls, precisions))
    return float(np.mean(aps)) if aps else 0.0


@torch.no_grad()
def validate(model, loader, image_size: int, conf_thresh: float, nms_thresh: float, device: torch.device, num_classes: int):
    model.eval()
    predictions, targets_for_metric = [], []
    iterator = tqdm(loader, desc="valid", leave=False) if tqdm else loader
    for images, targets in iterator:
        images = images.to(device)
        outputs = model(images)
        decoded = decode_detections(outputs, image_size, conf_thresh, nms_thresh, TOPK_CANDIDATES, MAX_DET)
        for pred, target in zip(decoded, targets):
            predictions.append({
                "image_id": target["image_id"],
                "boxes": pred["boxes"].cpu(),
                "scores": pred["scores"].cpu(),
                "labels": pred["labels"].cpu(),
            })
            targets_for_metric.append({
                "image_id": target["image_id"],
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu(),
            })
    return map_at_iou(predictions, targets_for_metric, num_classes=num_classes, iou_threshold=0.5)

## Train

In [8]:
train_ds = DetectionDataset(DATA_ROOT, "train", image_size=IMAGE_SIZE, augment=True, classes=CLASSES)
val_ds = DetectionDataset(DATA_ROOT, "val", image_size=IMAGE_SIZE, augment=False, classes=CLASSES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=DEVICE.type == "cuda")
val_loader = DataLoader(val_ds, batch_size=max(1, min(BATCH_SIZE, 16)), shuffle=False, num_workers=0, collate_fn=collate_fn, pin_memory=DEVICE.type == "cuda")

base_model = FCOSResNet50FPN(num_classes=len(CLASSES), pretrained=PRETRAINED_BACKBONE).to(DEVICE)
if GPU_COUNT > 1:
    print(f"using DataParallel on {GPU_COUNT} GPUs")
    model = nn.DataParallel(base_model)
else:
    model = base_model

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = max(1, len(train_loader))
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.15,
    div_factor=10,
    final_div_factor=100,
)
scaler = make_grad_scaler()


def model_state_dict_for_save(model):
    return model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()


best_map = -1.0
epochs_without_improve = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    iterator = tqdm(train_loader, desc=f"epoch {epoch:03d}") if tqdm else train_loader
    for step, (images, targets) in enumerate(iterator, start=1):
        images = images.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            outputs = model(images)
            losses = detection_loss(outputs, targets, image_size=IMAGE_SIZE)
            loss = losses["loss"]
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running += float(loss.detach().cpu())
        if tqdm:
            iterator.set_postfix(
                loss=running / step,
                cls=float(losses["cls_loss"]),
                box=float(losses["box_loss"]),
                ctr=float(losses["ctr_loss"]),
                pos=int(losses["num_pos"].item()),
            )
        elif step % 25 == 0 or step == len(train_loader):
            print(f"epoch={epoch:03d} step={step:04d}/{len(train_loader)} loss={running / step:.4f}")

    map50 = validate(model, val_loader, IMAGE_SIZE, VAL_CONF_THRESH, NMS_THRESH, DEVICE, len(CLASSES))
    print(f"epoch={epoch:03d} val_map50={map50:.4f}")
    checkpoint = {"model": model_state_dict_for_save(model), "classes": CLASSES, "image_size": IMAGE_SIZE, "epoch": epoch, "map50": map50}
    torch.save(checkpoint, RUN_DIR / "last.pt")
    if map50 > best_map:
        best_map = map50
        epochs_without_improve = 0
        torch.save(checkpoint, RUN_DIR / "best.pt")
        print(f"saved best checkpoint map50={best_map:.4f}")
    else:
        epochs_without_improve += 1

    if epoch >= MIN_EPOCHS and epochs_without_improve >= EARLY_STOP_PATIENCE:
        print(f"early stop: no improvement for {epochs_without_improve} epochs after epoch {epoch}")
        break

print("best val mAP@0.5:", best_map)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 213MB/s]


using DataParallel on 2 GPUs


epoch 001:   0%|          | 0/469 [00:00<?, ?it/s]

valid:   0%|          | 0/94 [00:00<?, ?it/s]

...

valid:   0%|          | 0/94 [00:00<?, ?it/s]

epoch=015 val_map50=0.7772


epoch 016:   0%|          | 0/469 [00:00<?, ?it/s]

valid:   0%|          | 0/94 [00:00<?, ?it/s]

epoch=016 val_map50=0.7776
best val mAP@0.5: 0.7844945054742981


## Sweep Thresholds

In [9]:
@torch.no_grad()
def load_eval_model(checkpoint_path: Path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    eval_model = FCOSResNet50FPN(num_classes=len(checkpoint.get("classes", CLASSES)), pretrained=False).to(DEVICE)
    eval_model.load_state_dict(checkpoint["model"])
    eval_model.eval()
    return eval_model, checkpoint


def sweep_thresholds(checkpoint_path: Path):
    eval_model, _ = load_eval_model(checkpoint_path)
    eval_loader = DataLoader(val_ds, batch_size=max(1, min(BATCH_SIZE, 16)), shuffle=False, num_workers=0, collate_fn=collate_fn, pin_memory=DEVICE.type == "cuda")
    best = (-1.0, None, None)
    for conf in [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15]:
        for nms_thr in [0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
            score = validate(eval_model, eval_loader, IMAGE_SIZE, conf, nms_thr, DEVICE, len(CLASSES))
            print(f"conf={conf:.3f} nms={nms_thr:.2f} map50={score:.4f}")
            if score > best[0]:
                best = (score, conf, nms_thr)
    print("best sweep:", best)
    return best


best_sweep = sweep_thresholds(RUN_DIR / "best.pt")
PRED_CONF_THRESH = best_sweep[1]
NMS_THRESH = best_sweep[2]
print("using for test:", PRED_CONF_THRESH, NMS_THRESH)

valid:   0%|          | 0/94 [00:00<?, ?it/s]

conf=0.005 nms=0.45 map50=0.7866


valid:   0%|          | 0/94 [00:00<?, ?it/s]

conf=0.005 nms=0.50 map50=0.7862


valid:   0%|          | 0/94 [00:00<?, ?it/s]

...

conf=0.150 nms=0.60 map50=0.7585


valid:   0%|          | 0/94 [00:00<?, ?it/s]

conf=0.150 nms=0.65 map50=0.7571


valid:   0%|          | 0/94 [00:00<?, ?it/s]

conf=0.150 nms=0.70 map50=0.7541
best sweep: (0.7871594475994824, 0.005, 0.55)
using for test: 0.005 0.55


## Predict Test with Horizontal Flip TTA

In [10]:
def letterbox_box_to_original(box: torch.Tensor, image_size: int, orig_h: int, orig_w: int, scale: float, pad_x: float, pad_y: float) -> dict[str, float]:
    x1, y1, x2, y2 = box.tolist()
    x1 = (x1 - pad_x) / scale
    x2 = (x2 - pad_x) / scale
    y1 = (y1 - pad_y) / scale
    y2 = (y2 - pad_y) / scale
    return {
        "x_min": round(max(0.0, min(float(orig_w - 1), x1)), 2),
        "y_min": round(max(0.0, min(float(orig_h - 1), y1)), 2),
        "x_max": round(max(0.0, min(float(orig_w - 1), x2)), 2),
        "y_max": round(max(0.0, min(float(orig_h - 1), y2)), 2),
    }


@torch.no_grad()
def predict_test(checkpoint_path: Path, output_json: Path, output_csv: Path, use_tta: bool = True):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    classes = checkpoint.get("classes", CLASSES)
    image_size = int(checkpoint.get("image_size", IMAGE_SIZE))
    pred_model = FCOSResNet50FPN(num_classes=len(classes), pretrained=False).to(DEVICE)
    pred_model.load_state_dict(checkpoint["model"])
    pred_model.eval()

    test_ds = InferenceImageDataset(DATA_ROOT, split="test", image_size=image_size)
    test_loader = DataLoader(test_ds, batch_size=max(1, min(BATCH_SIZE, 16)), shuffle=False, num_workers=0, collate_fn=collate_fn, pin_memory=DEVICE.type == "cuda")
    all_predictions = {}
    iterator = tqdm(test_loader, desc="test") if tqdm else test_loader
    for images, metas in iterator:
        images = images.to(DEVICE)
        decoded = decode_detections(pred_model(images), image_size, PRED_CONF_THRESH, NMS_THRESH, TOPK_CANDIDATES, MAX_DET)

        if use_tta:
            flipped = torch.flip(images, dims=[3])
            decoded_flip = decode_detections(pred_model(flipped), image_size, PRED_CONF_THRESH, NMS_THRESH, TOPK_CANDIDATES, MAX_DET)
            merged = []
            for normal, flip_pred in zip(decoded, decoded_flip):
                boxes_f = flip_pred["boxes"].clone()
                if len(boxes_f):
                    old_x1 = boxes_f[:, 0].clone()
                    old_x2 = boxes_f[:, 2].clone()
                    boxes_f[:, 0] = image_size - old_x2
                    boxes_f[:, 2] = image_size - old_x1
                boxes = torch.cat([normal["boxes"], boxes_f], dim=0)
                scores = torch.cat([normal["scores"], flip_pred["scores"]], dim=0)
                labels = torch.cat([normal["labels"], flip_pred["labels"]], dim=0)
                keep = classwise_nms(boxes, scores, labels, NMS_THRESH)[:MAX_DET] if len(boxes) else torch.empty((0,), dtype=torch.long, device=boxes.device)
                merged.append({"boxes": boxes[keep], "scores": scores[keep], "labels": labels[keep]})
            decoded = merged

        for pred, meta in zip(decoded, metas):
            orig_h, orig_w = [int(v) for v in meta["orig_size"]]
            scale, pad_x, pad_y = [float(v) for v in meta["scale_pad"]]
            rows = []
            for box, score, label in zip(pred["boxes"].cpu(), pred["scores"].cpu(), pred["labels"].cpu()):
                row = letterbox_box_to_original(box, image_size, orig_h, orig_w, scale, pad_x, pad_y)
                if row["x_max"] <= row["x_min"] or row["y_max"] <= row["y_min"]:
                    continue
                row["class"] = classes[int(label)]
                row["confidence"] = round(float(score), 5)
                rows.append(row)
            all_predictions[str(meta["image_id"])] = rows

    output_json.parent.mkdir(parents=True, exist_ok=True)
    with output_json.open("w", encoding="utf-8") as f:
        json.dump(all_predictions, f, indent=2)
    with output_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image_id", "bounding_boxes"])
        writer.writeheader()
        for image_id in sorted(all_predictions):
            writer.writerow({"image_id": image_id, "bounding_boxes": json.dumps(all_predictions[image_id])})
    print("wrote", output_json)
    print("wrote", output_csv)
    print("num test images:", len(all_predictions))
    print("first rows:", list(all_predictions.items())[:3])
    return all_predictions


predictions = predict_test(RUN_DIR / "best.pt", Path("/kaggle/working/predictions_fcos_v4.json"), Path("/kaggle/working/submission.csv"), use_tta=True)

test:   0%|          | 0/780 [00:00<?, ?it/s]

wrote /kaggle/working/predictions_fcos_v4.json
wrote /kaggle/working/submission.csv
num test images: 12474
first rows: [('img_0003a558b10b.jpg', [{'x_min': 8.23, 'y_min': 115.08, 'x_max': 353.02, 'y_max': 461.48, 'class': 'car', 'confidence': 0.62921}, {'x_min': 11.97, 'y_min': 100.72, 'x_max': 353.78, 'y_max': 249.12, 'class': 'car', 'confidence': 0.4345}, {'x_min': 73.81, 'y_min': 0.0, 'x_max': 338.53, 'y_max': 116.45, 'class': 'car', 'confidence': 0.30347}, {'x_min': 25.87, 'y_min': 5.0, 'x_max': 356.0, 'y_max': 209.71, 'class': 'car', 'confidence': 0.18999}, {'x_min': 0.0, 'y_min': 54.23, 'x_max': 347.92, 'y_max': 313.5, 'class': 'car', 'confidence': 0.14988}, {'x_min': 168.67, 'y_min': 9.48, 'x_max': 316.77, 'y_max': 77.21, 'class': 'car', 'confidence': 0.14074}, {'x_min': 172.22, 'y_min': 2.47, 'x_max': 302.11, 'y_max': 53.8, 'class': 'car', 'confidence': 0.10242}, {'x_min': 168.93, 'y_min': 13.87, 'x_max': 338.37, 'y_max': 111.65, 'class': 'car', 'confidence': 0.06485}, {'x_min'

## Practical Notes

- If this OOMs on T4 x2, set `BATCH_SIZE = 8`.
- This version intentionally trains fewer epochs because v3 peaked around epoch 8 and Kaggle killed the kernel later.
- If epoch 8-12 reaches 0.76+ validation, let the sweep and predict cells run; do not keep training manually.
- Submit `/kaggle/working/submission.csv`.